# 02.4 Regularization and Scheduler

This notebook focuses on two key questions in training:

1. how to reduce overfitting
2. how the learning rate changes over training

Key concepts:

- overfitting
- Dropout
- weight decay
- `StepLR`


## Learning Goals

After this notebook, you should be able to:

1. Recognize overfitting from training curves.
2. Understand how Dropout behaves differently in training and inference.
3. Understand the role of weight decay.
4. Use `StepLR` to change the learning rate.
5. Compare an unregularized model with a regularized one.
6. Transfer these ideas into larger future projects.

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

## Prepare the Data

To keep the focus on training behavior, we use the `digits` dataset and flatten images for an MLP.


In [ ]:
digits = load_digits()
X = torch.tensor(digits.images, dtype=torch.float32).reshape(-1, 64) / 16.0
y = torch.tensor(digits.target, dtype=torch.long)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

print("X_train.shape =", X_train.shape)
print("X_val.shape =", X_val.shape)

## Intuition for Dropout

The core idea of Dropout is to randomly "turn off" part of the neuron outputs during training.

The purpose is:

- reduce over-dependence between neurons
- improve generalization

In [ ]:
torch.manual_seed(0)

drop = nn.Dropout(p=0.5)
x = torch.ones(1, 8)

drop.train()
out_train_1 = drop(x)
out_train_2 = drop(x)

drop.eval()
out_eval_1 = drop(x)
out_eval_2 = drop(x)

print("train output 1 =", out_train_1)
print("train output 2 =", out_train_2)
print("eval output 1 =", out_eval_1)
print("eval output 2 =", out_eval_2)

Notice:

- outputs may differ each time in `train()` mode
- outputs are stable in `eval()` mode

This is one reason evaluation and inference must switch to `model.eval()`.


## An MLP with Optional Dropout

We use a small MLP to compare:

- no Dropout
- with Dropout

In [ ]:
class DigitsMLP(nn.Module):
    def __init__(self, dropout_p=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)


baseline_model = DigitsMLP(dropout_p=0.0)
regularized_model = DigitsMLP(dropout_p=0.3)

print(baseline_model)
print()
print(regularized_model)

## Training and Evaluation Functions

Here we reuse a simple training framework and also track the learning rate.


In [ ]:
def batch_accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_acc += batch_accuracy(logits, yb)
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches


def train_model(model, train_loader, val_loader, loss_fn, optimizer, scheduler=None, epochs=6):
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
        current_lr = optimizer.param_groups[0]["lr"]

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "lr": current_lr,
            }
        )

        if scheduler is not None:
            scheduler.step()

    return pd.DataFrame(history)

## Train an Unregularized Baseline

Setup here:

- no Dropout
- no weight decay
- no scheduler

In [ ]:
torch.manual_seed(0)
baseline = DigitsMLP(dropout_p=0.0)
loss_fn = nn.CrossEntropyLoss()
baseline_optimizer = torch.optim.Adam(baseline.parameters(), lr=0.01)

baseline_history = train_model(
    baseline,
    train_loader,
    val_loader,
    loss_fn,
    baseline_optimizer,
    scheduler=None,
    epochs=6,
)

print(baseline_history)

## Add Dropout, Weight Decay, and a Scheduler

This time the setup is:

- `dropout_p=0.3`
- `weight_decay=1e-3`
- `StepLR(step_size=3, gamma=0.5)`

In [ ]:
torch.manual_seed(0)
regularized = DigitsMLP(dropout_p=0.3)
regularized_optimizer = torch.optim.Adam(regularized.parameters(), lr=0.01, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(regularized_optimizer, step_size=3, gamma=0.5)

regularized_history = train_model(
    regularized,
    train_loader,
    val_loader,
    loss_fn,
    regularized_optimizer,
    scheduler=scheduler,
    epochs=6,
)

print(regularized_history)

## Compare Results

The point here is not to guarantee that one setup always wins, but to observe how these techniques affect training behavior.


In [ ]:
comparison = pd.DataFrame(
    {
        "model": ["baseline", "regularized"],
        "final_train_loss": [baseline_history.iloc[-1]["train_loss"], regularized_history.iloc[-1]["train_loss"]],
        "final_train_acc": [baseline_history.iloc[-1]["train_acc"], regularized_history.iloc[-1]["train_acc"]],
        "final_val_loss": [baseline_history.iloc[-1]["val_loss"], regularized_history.iloc[-1]["val_loss"]],
        "final_val_acc": [baseline_history.iloc[-1]["val_acc"], regularized_history.iloc[-1]["val_acc"]],
    }
)

print(comparison)

In [ ]:
print("baseline learning rates / baseline learning rates:")
print(baseline_history[["epoch", "lr"]])
print()
print("regularized learning rates / regularized learning rates:")
print(regularized_history[["epoch", "lr"]])

You should notice that:

- the baseline learning rate stays constant
- the regularized model reduces its learning rate after epoch 3

In [ ]:
# Exercise 1
# In one sentence, explain why Dropout behaves differently under train() and eval().


Exercise 1 Reference Answer

Dropout randomly masks activations during training to regularize the model, but it is disabled during evaluation so predictions are deterministic and use the full learned representation.

In [ ]:
# Exercise 2
# Change StepLR gamma from 0.5 to 0.1, then observe the learning-rate table.


Exercise 2 Reference Note

Changing `gamma` from `0.5` to `0.1` makes the learning rate drop much more aggressively at each scheduled step. This can stabilize late training, but it can also slow learning too early.

In [ ]:
# Exercise 3
# In one sentence, explain the intuition behind weight decay.


Exercise 3 Reference Answer

Weight decay discourages very large weights, which can reduce overfitting and usually makes the learned function smoother.

## Summary

The core of this notebook is not memorizing technique names, but understanding what problems they solve.

You should now be able to answer:

1. In what situations would you suspect overfitting?
2. Why does Dropout depend on `train()` and `eval()` modes?
3. What does weight decay mainly constrain?
4. Why does a scheduler change the learning rate instead of the loss directly?

Suggested next step:

- Move to the transfer-learning notebook and learn how to reuse an existing vision model structure.